In [1]:
# -*- coding: utf-8 -*-
"""
Поиск паттернов методом Кнута-Морриса-Пратта (КМП)
Google Colab версия с простыми комментариями
"""

def kmp_search(text, pattern):
    """
    Поиск всех вхождений паттерна в текст алгоритмом КМП

    Параметры:
        text (str): Текст, в котором ищем
        pattern (str): Паттерн (образец) для поиска

    Возвращает:
        list: Список индексов (позиций), где найден паттерн
    """
    if not pattern:
        return []

    # Шаг 1: Строим префикс-функцию для паттерна
    def build_prefix_function(p):
        """Префикс-функция: для каждой позиции - длина наибольшего префикса,
           который является суффиксом"""
        pi = [0] * len(p)
        j = 0  # длина текущего совпавшего префикса

        for i in range(1, len(p)):
            # Уменьшаем j, пока не найдем совпадение или j не станет 0
            while j > 0 and p[i] != p[j]:
                j = pi[j - 1]

            # Если символы совпали, увеличиваем j
            if p[i] == p[j]:
                j += 1

            pi[i] = j

        return pi

    # Строим префикс-функцию для нашего паттерна
    prefix = build_prefix_function(pattern)

    # Шаг 2: Поиск паттерна в тексте
    j = 0  # количество совпавших символов паттерна
    positions = []  # список найденных позиций

    for i in range(len(text)):
        # Если не совпадает, откатываем j по префикс-функции
        while j > 0 and text[i] != pattern[j]:
            j = prefix[j - 1]

        # Если символы совпали, увеличиваем j
        if text[i] == pattern[j]:
            j += 1

        # Если нашли полное совпадение (j == длина паттерна)
        if j == len(pattern):
            # Запоминаем позицию начала вхождения
            positions.append(i - j + 1)
            # Продолжаем поиск: откатываем j для поиска перекрывающихся вхождений
            j = prefix[j - 1]

    return positions


def search_multiple_patterns(text, patterns):
    """
    Поиск нескольких паттернов в тексте

    Параметры:
        text (str): Текст для поиска
        patterns (dict): Словарь {название_паттерна: паттерн}

    Возвращает:
        dict: Результаты поиска для каждого паттерна
    """
    results = {}

    for name, pattern in patterns.items():
        positions = kmp_search(text, pattern)
        if positions:
            results[name] = {
                'pattern': pattern,
                'positions': positions,
                'count': len(positions)
            }

    return results


# ==================== СОЗДАНИЕ ТЕСТОВЫХ ДАННЫХ ====================

# Создаем тестовый текст
test_text = """
Иванов Иван Иванович родился 15.03.1985 года в городе Москва.
Его СНИЛС: 123-456-789 01, а паспорт: 4510 123456.
Контактный email: ivan.ivanov@example.com.
Также в документе упоминается другой email: i.ivanov@work.ru.
СНИЛС сотрудника: 987-654-321 00.
Дата рождения: 15.03.1985, также есть дата 20.12.1990.
Телефон: +7-903-123-45-67.
Еще один номер: 8 (495) 123-45-67.
"""

# Определяем паттерны для поиска (простые, для демонстрации)
patterns_to_find = {
    "СНИЛС": r"123-456-789 01",
    "Email_1": r"ivan.ivanov@example.com",
    "Email_2": r"i.ivanov@work.ru",
    "Дата_рождения": r"15\.03\.1985",
    "Телефон1": r"\+7-903-123-45-67",
    "Телефон2": r"8 \(495\) 123-45-67",
    "Слово_Москва": r"Москва",
    "Имя_Иван": r"Иван"
}

# Дополнительный текст с перекрывающимися паттернами (для демонстрации работы КМП)
test_text_overlap = """
aaaaaa
Поиск паттерна 'aa' в строке 'aaaaaa' покажет позиции: 0,1,2,3,4
Проверка перекрывающихся вхождений: ababa (паттерн 'aba')
"""

patterns_overlap = {
    "Два_а": "aa",
    "АБА": "aba"
}


# ==================== ЗАПУСК ПОИСКА ====================

print("="*60)
print("ПОИСК ПАТТЕРНОВ МЕТОДОМ КНУТА-МОРРИСА-ПРАТТА (КМП)")
print("="*60)

# Поиск в основном тексте
print("\n1. Анализ основного текста:")
print("-"*40)
print(f"Текст для анализа:\n{test_text}")
print("-"*40)

results = search_multiple_patterns(test_text, patterns_to_find)

if results:
    print("\nРЕЗУЛЬТАТЫ ПОИСКА:")
    for name, info in results.items():
        print(f"\n▶ Паттерн '{name}': '{info['pattern']}'")
        print(f"   Найден {info['count']} раз(а) на позициях: {info['positions']}")

        # Показываем контекст первого найденного совпадения
        if info['positions']:
            pos = info['positions'][0]
            start = max(0, pos - 10)
            end = min(len(test_text), pos + len(info['pattern']) + 10)
            context = test_text[start:end].replace('\n', ' ')
            print(f"   Контекст: ...{context}...")
else:
    print("\nПаттерны не найдены.")

# Демонстрация работы с перекрывающимися паттернами
print("\n\n" + "="*60)
print("ДЕМОНСТРАЦИЯ ПЕРЕКРЫВАЮЩИХСЯ ВХОЖДЕНИЙ")
print("="*60)

print("\n2. Тест с перекрывающимися паттернами:")
print("-"*40)
print(f"Текст: 'aaaaaa'")
print(f"Поиск паттерна 'aa'")

positions = kmp_search("aaaaaa", "aa")
print(f"Результат: паттерн 'aa' найден на позициях: {positions}")
print("Пояснение: КМП находит ВСЕ вхождения, включая перекрывающиеся (0-1, 1-2, 2-3, 3-4, 4-5)")

print("\n" + "-"*40)
print(f"Текст: 'ababa'")
print(f"Поиск паттерна 'aba'")

positions = kmp_search("ababa", "aba")
print(f"Результат: паттерн 'aba' найден на позициях: {positions}")
print("Пояснение: найдены позиции 0 (aba) и 2 (aba) - перекрывающиеся вхождения")


# ==================== ФУНКЦИЯ ДЛЯ ВИЗУАЛЬНОЙ ДЕМОНСТРАЦИИ ====================

def visualize_kmp_search(text, pattern):
    """
    Визуализирует процесс поиска КМП с подсветкой найденных паттернов
    """
    positions = kmp_search(text, pattern)

    print(f"\nТекст: {text}")
    print(f"Паттерн: {pattern}")
    print(f"Найден на позициях: {positions}")

    # Визуализация
    result_chars = list(text)
    marker = [' '] * len(text)

    for pos in positions:
        for i in range(len(pattern)):
            if pos + i < len(text):
                marker[pos + i] = '^'

    print(f"Разметка: {''.join(marker)}")
    print(f"Количество вхождений: {len(positions)}")


# Дополнительная визуализация
print("\n\n" + "="*60)
print("ВИЗУАЛИЗАЦИЯ ПОИСКА")
print("="*60)

visualize_kmp_search("ABABABABAB", "ABAB")
visualize_kmp_search("привет мир привет мир", "привет")
visualize_kmp_search("abcdeabcdeabcde", "abcde")

ПОИСК ПАТТЕРНОВ МЕТОДОМ КНУТА-МОРРИСА-ПРАТТА (КМП)

1. Анализ основного текста:
----------------------------------------
Текст для анализа:

Иванов Иван Иванович родился 15.03.1985 года в городе Москва.
Его СНИЛС: 123-456-789 01, а паспорт: 4510 123456.
Контактный email: ivan.ivanov@example.com.
Также в документе упоминается другой email: i.ivanov@work.ru.
СНИЛС сотрудника: 987-654-321 00.
Дата рождения: 15.03.1985, также есть дата 20.12.1990.
Телефон: +7-903-123-45-67.
Еще один номер: 8 (495) 123-45-67.

----------------------------------------

РЕЗУЛЬТАТЫ ПОИСКА:

▶ Паттерн 'СНИЛС': '123-456-789 01'
   Найден 1 раз(а) на позициях: [74]
   Контекст: ...го СНИЛС: 123-456-789 01, а паспор...

▶ Паттерн 'Email_1': 'ivan.ivanov@example.com'
   Найден 1 раз(а) на позициях: [132]
   Контекст: ...ый email: ivan.ivanov@example.com. Также в ...

▶ Паттерн 'Email_2': 'i.ivanov@work.ru'
   Найден 1 раз(а) на позициях: [201]
   Контекст: ...ой email: i.ivanov@work.ru. СНИЛС со...

▶ Паттерн 'Слов